# Data Cleaning

In [2]:
import pandas as pd
import os
import ast
from pathlib import Path

In [ ]:
etl_df = pd.read_csv("../data/raw/anime_data.csv")
current_df = pd.read_csv("../data/raw/current_data.csv")

df = pd.concat([etl_df, current_df])

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 72 non-null     int64  
 1   title                  72 non-null     str    
 2   source                 72 non-null     str    
 3   episodes               1 non-null      float64
 4   synopsis               72 non-null     str    
 5   year                   72 non-null     int64  
 6   season                 72 non-null     str    
 7   producers              72 non-null     str    
 8   genres                 72 non-null     str    
 9   studios                72 non-null     str    
 10  demographics           72 non-null     str    
 11  themes                 72 non-null     str    
 12  rating                 31 non-null     str    
 13  sequel                 72 non-null     bool   
 14  favorites              72 non-null     int64  
 15  score              

Let's remove duplicates.

In [ ]:
df2 = df.drop_duplicates(subset=['mal_id'], keep='first')
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 72 non-null     int64  
 1   title                  72 non-null     str    
 2   source                 72 non-null     str    
 3   episodes               1 non-null      float64
 4   synopsis               72 non-null     str    
 5   year                   72 non-null     int64  
 6   season                 72 non-null     str    
 7   producers              72 non-null     str    
 8   genres                 72 non-null     str    
 9   studios                72 non-null     str    
 10  demographics           72 non-null     str    
 11  themes                 72 non-null     str    
 12  rating                 31 non-null     str    
 13  sequel                 72 non-null     bool   
 14  favorites              72 non-null     int64  
 15  score              

## Future Anime

In the data collection process, one of the criteria was that it should not be currently airing. This will of course include Fall 2026 and future anime. We will remove anime with release year 2027 and above, since we need Fall 2026 anime as our final prediction data.

In [ ]:
df3 = df2[df2['year'] >= 2027]

<class 'pandas.DataFrame'>
Index: 8404 entries, 0 to 8845
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        8404 non-null   int64  
 1   title         8404 non-null   str    
 2   source        8404 non-null   str    
 3   episodes      8404 non-null   float64
 4   synopsis      6068 non-null   str    
 5   year          6405 non-null   float64
 6   season        6405 non-null   str    
 7   producers     8404 non-null   str    
 8   genres        8404 non-null   str    
 9   studios       8404 non-null   str    
 10  demographics  8404 non-null   str    
 11  themes        8404 non-null   str    
 12  rating        8337 non-null   str    
 13  sequel        8404 non-null   bool   
 14  favorites     8404 non-null   int64  
 15  score         5411 non-null   float64
 16  wc            8404 non-null   int64  
 17  dropped       8404 non-null   int64  
 18  forum         8404 non-null   int64  
 19  t

## Synopsis

No synopsis should be fine. My justification is that shows with no synopsis might be boring for users who look at MAL. However, categorizing by age rating is important: we need to know if restrictive shows have lower metrics.

In [ ]:
df4 = df3[~df3['rating'].isna()]
df4.info()

<class 'pandas.DataFrame'>
Index: 5338 entries, 0 to 8816
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        5338 non-null   int64  
 1   title         5338 non-null   str    
 2   source        5338 non-null   str    
 3   episodes      5338 non-null   float64
 4   synopsis      5302 non-null   str    
 5   year          5338 non-null   float64
 6   season        5338 non-null   str    
 7   producers     5338 non-null   str    
 8   genres        5338 non-null   str    
 9   studios       5338 non-null   str    
 10  demographics  5338 non-null   str    
 11  themes        5338 non-null   str    
 12  rating        5338 non-null   str    
 13  sequel        5338 non-null   bool   
 14  favorites     5338 non-null   int64  
 15  score         5338 non-null   float64
 16  wc            5338 non-null   int64  
 17  dropped       5338 non-null   int64  
 18  forum         5338 non-null   int64  
 19  t

Additionally, we can fill the null synopsis entries with an empty string for data entry purposes.

In [ ]:
df4.fillna({'synopsis': " "}, inplace=True)
df4.info()

<class 'pandas.DataFrame'>
Index: 5338 entries, 0 to 8816
Data columns (total 20 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mal_id        5338 non-null   int64  
 1   title         5338 non-null   str    
 2   source        5338 non-null   str    
 3   episodes      5338 non-null   float64
 4   synopsis      5338 non-null   str    
 5   year          5338 non-null   float64
 6   season        5338 non-null   str    
 7   producers     5338 non-null   str    
 8   genres        5338 non-null   str    
 9   studios       5338 non-null   str    
 10  demographics  5338 non-null   str    
 11  themes        5338 non-null   str    
 12  rating        5338 non-null   str    
 13  sequel        5338 non-null   bool   
 14  favorites     5338 non-null   int64  
 15  score         5338 non-null   float64
 16  wc            5338 non-null   int64  
 17  dropped       5338 non-null   int64  
 18  forum         5338 non-null   int64  
 19  t

## Multi-valued Features

If we check some multi-valued features such as genres, we can see that they're not actually lists, but strings.

In [ ]:
df4['genres']

0                ['Action', 'Award Winning', 'Sci-Fi']
1                    ['Action', 'Adventure', 'Sci-Fi']
2       ['Action', 'Drama', 'Mystery', 'Supernatural']
3                   ['Action', 'Adventure', 'Fantasy']
4                                           ['Sports']
                             ...                      
8719                 ['Fantasy', 'Gourmet', 'Romance']
8726                                       ['Romance']
8735                ['Action', 'Adventure', 'Fantasy']
8741                             ['Comedy', 'Romance']
8816                        ['Comedy', 'Supernatural']
Name: genres, Length: 5338, dtype: str

Let's turn them into actual lists.

In [ ]:
def parse_list_col(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):  # already a real list, don't double-parse
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

df4[['themes', 'genres', 'studios', 'demographics', 'producers']] = df4[['themes', 'genres', 'studios', 'demographics', 'producers']].map(parse_list_col)
df4['themes']

0                   [Historical, Medical]
1                                      []
2                 [Isekai, Reincarnation]
3     [Love Polygon, School, Team Sports]
4              [Delinquents, Time Travel]
                     ...                 
67                                     []
68                                     []
69                                     []
70                            [Gag Humor]
71                      [Anthropomorphic]
Name: themes, Length: 72, dtype: object

For now, we can save this.

In [ ]:
# target_dir = Path("../data/processed")
# file_path = target_dir / "anime_data_1.parquet"
# df4.to_parquet(file_path, engine='pyarrow')

target_dir = Path("../data/processed")
file_path = target_dir / "anime_data_1.parquet"
df4.to_parquet(file_path, engine='pyarrow')